In [1]:
# =============================================================================
# GNN Pipeline for Anti-inflammatory Activity Prediction (FINAL CLEAN VERSION)
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import (
    GINEConv, GATv2Conv, TransformerConv,
    global_mean_pool, global_max_pool,
    GraphNorm, GlobalAttention
)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR
from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.data import WeightedRandomSampler
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, matthews_corrcoef, recall_score,
    confusion_matrix, balanced_accuracy_score, f1_score,
    precision_score, r2_score, mean_squared_error, mean_absolute_error,
    roc_curve
)
from sklearn.model_selection import KFold, train_test_split
from scipy import stats
from rdkit import RDLogger, Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, QED, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
import warnings, copy, optuna, os
 
warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')
optuna.logging.set_verbosity(optuna.logging.WARNING)

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [3]:
# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
PALETTE = {"C1_noaug": "#4C72B0", "C2_aug": "#55A868",
           "C3_noaug": "#C44E52", "C4_aug": "#8172B2"}
FIG_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__))
                        if '__file__' in dir() else ".", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
 
def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  [saved] {path}")

In [4]:
# =============================================================================
# ATOM & BOND FEATURES
# =============================================================================
def enhanced_atom_features(atom):
    """21-dim node feature vector covering all required features."""
    atom_types = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P', 'B']
    features = [1.0 if atom.GetSymbol() == t else 0.0 for t in atom_types]
    features += [
        atom.GetAtomicNum() / 100.0,          # atomic number
        atom.GetDegree() / 8.0,               # degree
        atom.GetFormalCharge() / 5.0,         # formal charge
        atom.GetTotalNumHs() / 8.0,           # hydrogen count
        atom.GetTotalValence() / 8.0,         # valence
        float(atom.GetIsAromatic()),           # aromaticity
        float(atom.IsInRing()),               # ring membership
        float(atom.GetChiralTag() != Chem.rdchem.ChiralType.CHI_UNSPECIFIED)  # chirality
    ]
    hyb = atom.GetHybridization()
    for ht in [Chem.rdchem.HybridizationType.SP,
               Chem.rdchem.HybridizationType.SP2,
               Chem.rdchem.HybridizationType.SP3]:
        features.append(1.0 if hyb == ht else 0.0)  # hybridization (one-hot)
    return features  # 21 dims
 
 
def enhanced_bond_features(bond):
    """7-dim edge feature vector."""
    bt = bond.GetBondType()
    return [
        float(bt == Chem.rdchem.BondType.SINGLE),
        float(bt == Chem.rdchem.BondType.DOUBLE),
        float(bt == Chem.rdchem.BondType.TRIPLE),
        float(bt == Chem.rdchem.BondType.AROMATIC),
        float(bond.GetIsConjugated()),
        float(bond.IsInRing()),
        float(bond.GetStereo() != Chem.rdchem.BondStereo.STEREONONE)
    ]  # 7 dims

In [5]:
# =============================================================================
# SMILES → GRAPH
# =============================================================================
def smiles_to_graph(smiles, label, pic50):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    x = [enhanced_atom_features(a) for a in mol.GetAtoms()]
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        f = enhanced_bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr  += [f, f]
    if len(edge_index) == 0:
        edge_index = [[0, 0], [0, 0]]
        edge_attr  = [[0.0] * 7, [0.0] * 7]
    try:
        desc = [
            Descriptors.MolWt(mol) / 1000.0,
            Descriptors.MolLogP(mol) / 10.0,
            Descriptors.TPSA(mol) / 200.0,
            Descriptors.NumRotatableBonds(mol) / 20.0,
            QED.qed(mol),
            Descriptors.NumHDonors(mol) / 10.0,
            Descriptors.NumHAcceptors(mol) / 10.0,
            float(rdMolDescriptors.CalcNumAromaticRings(mol)) / 5.0,
            Descriptors.FractionCSP3(mol),
            float(mol.GetNumHeavyAtoms()) / 50.0,
            float(rdMolDescriptors.CalcNumRings(mol)) / 10.0,
            min(Descriptors.BertzCT(mol) / 1000.0, 3.0),
            float(rdMolDescriptors.CalcNumHeteroatoms(mol)) / 20.0,
        ]
    except Exception:
        desc = [0.0] * 13
    return Data(
        x=torch.tensor(x, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float),
        desc=torch.tensor(desc, dtype=torch.float).view(1, -1),
        y_cls=torch.tensor([label], dtype=torch.float),
        y_reg=torch.tensor([pic50], dtype=torch.float),
        smiles=smiles)

In [6]:
# =============================================================================
# LOAD DATA
# =============================================================================
df = pd.read_csv(r"D:\Biotechnology Research\WholedatasetGAN.csv")
df["active"] = (df["pIC50"] >= 6).astype(int)
df = df.dropna(subset=["pIC50"]).drop_duplicates(subset="SMILES").reset_index(drop=True)
 
graphs, valid_smiles = [], []
for _, row in df.iterrows():
    g = smiles_to_graph(row.SMILES, row.active, row.pIC50)
    if g is not None:
        graphs.append(g)
        valid_smiles.append(row.SMILES)
 
df_filtered = df[df['SMILES'].isin(valid_smiles)].reset_index(drop=True)
y_reg_all  = np.array([g.y_reg.item() for g in graphs])
y_reg_mean = y_reg_all.mean()
y_reg_std  = y_reg_all.std() + 1e-8
for g in graphs:
    g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
 
NODE_DIM = graphs[0].x.shape[1]
DESC_DIM = graphs[0].desc.shape[1]
class_counts  = df_filtered['active'].value_counts()
n_total       = len(df_filtered)
n_pos         = class_counts.get(1, 1)
n_neg         = class_counts.get(0, 1)
pos_weight_val = n_total / (2.0 * n_pos)
neg_weight_val = n_total / (2.0 * n_neg)
 
print(f"Dataset: {len(graphs)} graphs | node_dim={NODE_DIM} | desc_dim={DESC_DIM}")
print(f"Class balance: {class_counts.to_dict()}")
print(f"pIC50 mean={y_reg_mean:.3f}  std={y_reg_std:.3f}")

Dataset: 4300 graphs | node_dim=21 | desc_dim=13
Class balance: {0: 2150, 1: 2150}
pIC50 mean=6.408  std=1.208


In [7]:
# =============================================================================
# CHEMICAL SPACE ANALYSIS — TANIMOTO
# =============================================================================
def tanimoto_analysis(smiles_list, sample_size=1000):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list if Chem.MolFromSmiles(s)]
    fps  = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048)
            for m in mols[:sample_size]]
    sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j])
            for i in range(len(fps)) for j in range(i + 1, len(fps))]
    mean_sim, std_sim = np.mean(sims), np.std(sims)
    print(f"Tanimoto (n={len(fps)}): Mean={mean_sim:.4f}±{std_sim:.4f}  "
          f"Diversity={1-mean_sim:.4f}")
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(sims, bins=60, color="#4C72B0", edgecolor='white', alpha=0.85)
    axes[0].axvline(mean_sim, color='red', linestyle='--', linewidth=1.5,
                    label=f'Mean = {mean_sim:.3f}')
    axes[0].set_xlabel("Tanimoto Similarity")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title("Pairwise Tanimoto Similarity Distribution")
    axes[0].legend()
 
    # pIC50 distribution
    axes[1].hist(df_filtered['pIC50'], bins=50, color="#55A868",
                 edgecolor='white', alpha=0.85)
    axes[1].axvline(6.0, color='red', linestyle='--', linewidth=1.5,
                    label='Activity threshold (6.0)')
    axes[1].set_xlabel("pIC50")
    axes[1].set_ylabel("Count")
    axes[1].set_title("pIC50 Distribution")
    axes[1].legend()
    plt.suptitle("Chemical Space Analysis", fontweight='bold', y=1.02)
    plt.tight_layout()
    savefig("01_chemical_space.png")
    return mean_sim, std_sim, 1 - mean_sim
 
print("\n--- Chemical Space Analysis ---")
tanimoto_analysis(df_filtered['SMILES'].tolist())


--- Chemical Space Analysis ---
Tanimoto (n=1000): Mean=0.1170±0.0619  Diversity=0.8830
  [saved] .\figures\01_chemical_space.png


(np.float64(0.1169745449806943),
 np.float64(0.06189532492100967),
 np.float64(0.8830254550193057))

In [8]:
# =============================================================================
# FOCAL LOSS
# =============================================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.6, gamma=2.0, label_smoothing=0.03):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
 
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt  = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

In [9]:
# =============================================================================
# AUGMENTATION
# =============================================================================
def edge_dropout(data, p=0.15):
    data = data.clone()
    n = data.edge_index.size(1) // 2
    if n == 0:
        return data
    mask = torch.rand(n, device=data.x.device) > p
    mask = mask.repeat_interleave(2)
    data.edge_index = data.edge_index[:, mask]
    data.edge_attr  = data.edge_attr[mask]
    return data
 
def node_dropout(data, p=0.1):
    data = data.clone()
    mask = torch.rand(data.x.size(0), 1, device=data.x.device) > p
    data.x = data.x * mask
    return data
 
def smiles_augmentation(smiles, label, pic50, n_aug=3):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []
    aug = []
    for _ in range(n_aug):
        try:
            rand_smi = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
            g = smiles_to_graph(rand_smi, label, pic50)
            if g is not None:
                g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
                aug.append(g)
        except Exception:
            continue
    return aug
 
print("\nPrecomputing augmentation cache (n_aug=3)...")
aug_cache = {}
for idx in range(len(graphs)):
    row = df_filtered.iloc[idx]
    aug_cache[idx] = smiles_augmentation(
        row['SMILES'], row['active'], row['pIC50'], n_aug=3)
total_aug = sum(len(v) for v in aug_cache.values())
print(f"Cache ready: {total_aug} augmented graphs stored.")


Precomputing augmentation cache (n_aug=3)...
Cache ready: 12900 augmented graphs stored.


In [10]:
# =============================================================================
# TEST-TIME AUGMENTATION (TTA)
# =============================================================================
def tta_predict(model, smiles_list, labels_cls, labels_reg,
                device, n_tta=4, batch_size=192):
    model.eval()
    all_probs = [[] for _ in range(len(smiles_list))]
    all_regs  = [[] for _ in range(len(smiles_list))]
    for t in range(n_tta + 1):
        tta_graphs, tta_idx = [], []
        for i, (smi, lc, lr) in enumerate(zip(smiles_list, labels_cls, labels_reg)):
            if t == 0:
                g = smiles_to_graph(smi, int(lc), float(lr))
            else:
                mol = Chem.MolFromSmiles(smi)
                if mol is None: continue
                rand_smi = Chem.MolToSmiles(mol, doRandom=True, canonical=False)
                g = smiles_to_graph(rand_smi, int(lc), float(lr))
            if g is not None:
                g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
                tta_graphs.append(g)
                tta_idx.append(i)
        loader = DataLoader(tta_graphs, batch_size=batch_size,
                            shuffle=False, num_workers=0)
        ptr = 0
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(device)
                co, ro = model(batch)
                ps = torch.sigmoid(co).cpu().numpy()
                rs = ro.cpu().numpy()
                for k in range(len(ps)):
                    all_probs[tta_idx[ptr + k]].append(float(ps[k]))
                    all_regs [tta_idx[ptr + k]].append(float(rs[k]))
                ptr += len(ps)
    avg_probs = np.array([np.mean(p) if p else 0.5 for p in all_probs])
    avg_regs  = np.array([np.mean(r) if r else 0.0 for r in all_regs])
    return avg_probs, avg_regs

In [11]:
# =============================================================================
# WEIGHTED SAMPLER
# =============================================================================
def make_weighted_sampler(graph_list):
    weights = []
    for g in graph_list:
        lbl = int(g.y_cls.item())
        weights.append(pos_weight_val if lbl == 1 else neg_weight_val)
    weights = torch.tensor(weights, dtype=torch.float)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

In [12]:
# =============================================================================
# MODEL — High-Performance MSMP
# =============================================================================
class HighPerfMSMP(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=192, heads=6,
                 dropout=0.25, num_layers=3):
        super().__init__()
        self.node_embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.edge_proj_gine = nn.Linear(7, hidden)
        self.edge_proj_attn = nn.Linear(7, 32)
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        self.layers = nn.ModuleList()
        self.residual_scales = nn.ParameterList()
        self.num_layers = num_layers
        for i in range(num_layers):
            dp_i = dropout * (1 + 0.1 * i)
            self.layers.append(nn.ModuleDict({
                'gine': GINEConv(nn.Sequential(
                    nn.Linear(hidden, hidden * 2), nn.GELU(),
                    nn.Dropout(dp_i), nn.Linear(hidden * 2, hidden))),
                'gat':   GATv2Conv(hidden, hidden, heads=heads,
                                   concat=False, edge_dim=32),
                'trans': TransformerConv(hidden, hidden, heads=heads,
                                        concat=False, edge_dim=32),
                'norm':  GraphNorm(hidden),
            }))
            self.residual_scales.append(nn.Parameter(torch.tensor(0.5)))
        self.layer_pool_weights = nn.Parameter(
            torch.ones(num_layers) / num_layers)
        gate_nn = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1))
        self.attn_pool = GlobalAttention(gate_nn)
        pool_dim = hidden * 4 + desc_dim
        self.pool_norm = nn.LayerNorm(pool_dim)
        self.fusion = nn.Sequential(
            nn.Linear(pool_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.GELU(),
            nn.Dropout(dropout * 0.5), nn.Linear(hidden // 2, 128))
        self.cls_head = nn.Sequential(
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, 1))
        self.reg_head = nn.Sequential(
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, 1))
 
    def forward(self, data):
        x   = self.node_embed(data.x)
        ei, ea, bat = data.edge_index, data.edge_attr, data.batch
        desc = data.desc
        if desc.size(0) > 1:
            desc = self.desc_bn(desc)
        e_gine = self.edge_proj_gine(ea)
        e_attn = self.edge_proj_attn(ea)
        layer_pool = []
        for i, layer in enumerate(self.layers):
            h = (layer['gine'](x, ei, e_gine) +
                 layer['gat'](x, ei, e_attn)  +
                 layer['trans'](x, ei, e_attn))
            h = layer['norm'](h, bat)
            scale = torch.sigmoid(self.residual_scales[i])
            x = F.gelu(x + scale * h)
            layer_pool.append(global_mean_pool(h, bat))
        h_mean  = global_mean_pool(x, bat)
        h_max   = global_max_pool(x, bat)
        h_attn  = self.attn_pool(x, bat)
        lw      = F.softmax(self.layer_pool_weights, dim=0)
        h_layer = (torch.stack(layer_pool, dim=1) *
                   lw.view(1, -1, 1)).sum(dim=1)
        g = torch.cat([h_mean, h_max, h_attn, h_layer, desc], dim=-1)
        g = self.pool_norm(g)
        f = self.fusion(g)
        return self.cls_head(f).squeeze(-1), self.reg_head(f).squeeze(-1)

In [13]:
# =============================================================================
# LR SCHEDULER  (warmup → cosine, no lr conflict)
# =============================================================================
def build_scheduler(optimizer, lr, warmup_epochs=5):
    warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                      total_iters=warmup_epochs)
    cosine = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)
    return SequentialLR(optimizer, schedulers=[warmup, cosine],
                        milestones=[warmup_epochs])

In [14]:
# =============================================================================
# TRAINING
# =============================================================================
def train_epoch(model, loader, optimizer, scheduler, hparams,
                device, epoch, verbose=True, use_aug=True):
    model.train()
    total = 0.0
    cls_fn = FocalLoss(alpha=hparams['focal_alpha'],
                       gamma=hparams['focal_gamma'],
                       label_smoothing=hparams.get('label_smoothing', 0.03))
    for batch in loader:
        batch = batch.to(device)
        if use_aug:
            if torch.rand(1) > 0.3:
                batch = edge_dropout(batch, hparams['edge_dropout'])
            batch = node_dropout(batch, hparams['node_dropout'])
        optimizer.zero_grad()
        cls_out, reg_out = model(batch)
        cls_loss = cls_fn(cls_out, batch.y_cls.squeeze())
        reg_loss = F.huber_loss(
            reg_out, batch.y_reg_norm.squeeze(), delta=1.0)
        loss = cls_loss + hparams['reg_weight'] * reg_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += loss.item()
    scheduler.step()
    if verbose and (epoch % 5 == 0 or epoch == 0):
        lr_now = optimizer.param_groups[0]['lr']
        print(f"    Epoch {epoch:3d}: loss={total/len(loader):.4f}  "
              f"lr={lr_now:.2e}")
    return total / len(loader)

In [15]:
# =============================================================================
# VALIDATE
# =============================================================================
def validate_model_acc(model, loader, device):
    """Returns (best_acc, auc, best_thr, probs, labels)."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            co, _ = model(batch)
            probs.extend(torch.sigmoid(co).cpu().numpy())
            labels.extend(batch.y_cls.cpu().numpy())
    probs  = np.array(probs)
    labels = np.array(labels)
    thrs = np.linspace(0.1, 0.9, 161)
    accs = [accuracy_score(labels, (probs >= t).astype(int)) for t in thrs]
    best_t   = float(thrs[np.argmax(accs)])
    best_acc = float(max(accs))
    try:
        auc = float(roc_auc_score(labels, probs))
    except Exception:
        auc = 0.5
    return best_acc, auc, best_t, probs, labels

In [16]:
# =============================================================================
# HYPERPARAMETER TUNING (Optuna)  — shared across all conditions
# =============================================================================
def run_optuna(n_trials=30):
    def objective(trial):
        hp = {
            'lr':              trial.suggest_float('lr', 2e-4, 8e-4, log=True),
            'hidden':          trial.suggest_categorical('hidden', [160, 192, 224]),
            'heads':           trial.suggest_categorical('heads', [4, 6, 8]),
            'dropout':         trial.suggest_float('dropout', 0.18, 0.32),
            'weight_decay':    trial.suggest_float('weight_decay', 1e-5, 5e-5, log=True),
            'edge_dropout':    trial.suggest_float('edge_dropout', 0.08, 0.20),
            'node_dropout':    trial.suggest_float('node_dropout', 0.04, 0.12),
            'focal_alpha':     trial.suggest_float('focal_alpha', 0.45, 0.75),
            'focal_gamma':     trial.suggest_float('focal_gamma', 1.8, 2.6),
            'reg_weight':      trial.suggest_float('reg_weight', 0.05, 0.20),
            'label_smoothing': trial.suggest_float('label_smoothing', 0.0, 0.08),
        }
        val_seed = trial.number % 5
        idx = list(range(len(graphs)))
        train_idx, val_idx = train_test_split(
            idx, test_size=0.2, random_state=val_seed,
            stratify=df_filtered['active'].values)
        train_graphs = []
        for i in train_idx:
            train_graphs.append(graphs[i])
            train_graphs.extend(aug_cache[i])
        sampler = make_weighted_sampler(train_graphs)
        train_loader = DataLoader(train_graphs, batch_size=192,
                                  sampler=sampler, num_workers=0)
        val_loader   = DataLoader([graphs[i] for i in val_idx],
                                  batch_size=192, shuffle=False, num_workers=0)
        model = HighPerfMSMP(
            NODE_DIM, DESC_DIM,
            **{k: hp[k] for k in ['hidden', 'heads', 'dropout']}
        ).to(device)
        opt   = AdamW(model.parameters(), lr=hp['lr'],
                      weight_decay=hp['weight_decay'])
        sched = build_scheduler(opt, hp['lr'], warmup_epochs=5)
        best_acc, wait = 0.0, 0
        for epoch in range(50):
            train_epoch(model, train_loader, opt, sched, hp,
                        device, epoch, verbose=False, use_aug=True)
            val_acc, _, _, _, _ = validate_model_acc(
                model, val_loader, device)
            if val_acc > best_acc:
                best_acc = val_acc
                wait = 0
            else:
                wait += 1
                if wait >= 10:
                    break
        return best_acc
 
    def cb(study, trial):
        print(f"  Trial {trial.number:2d}: acc={trial.value:.4f}  "
              f"(best={study.best_value:.4f})")
 
    print("\n🎯 HYPERPARAMETER TUNING (30 trials)...")
    study = optuna.create_study(
        direction="maximize",
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
    study.optimize(objective, n_trials=n_trials, callbacks=[cb])
    print("\n🏆 BEST HYPERPARAMS:", study.best_trial.params)
    return study.best_trial.params
 
 
# Run Optuna (or use cached best_hparams if already run)
best_hparams = run_optuna(n_trials=30)


🎯 HYPERPARAMETER TUNING (30 trials)...
  Trial  0: acc=0.8919  (best=0.8919)
  Trial  1: acc=0.8965  (best=0.8965)
  Trial  2: acc=0.8849  (best=0.8965)
  Trial  3: acc=0.9000  (best=0.9000)
  Trial  4: acc=0.8756  (best=0.9000)
  Trial  5: acc=0.8802  (best=0.9000)
  Trial  6: acc=0.8965  (best=0.9000)
  Trial  7: acc=0.8884  (best=0.9000)
  Trial  8: acc=0.8895  (best=0.9000)
  Trial  9: acc=0.8942  (best=0.9000)
  Trial 10: acc=0.8884  (best=0.9000)
  Trial 11: acc=0.8930  (best=0.9000)
  Trial 12: acc=0.8826  (best=0.9000)
  Trial 13: acc=0.8953  (best=0.9000)
  Trial 14: acc=0.8709  (best=0.9000)
  Trial 15: acc=0.8872  (best=0.9000)
  Trial 16: acc=0.8849  (best=0.9000)
  Trial 17: acc=0.8826  (best=0.9000)
  Trial 18: acc=0.8953  (best=0.9000)
  Trial 19: acc=0.8663  (best=0.9000)
  Trial 20: acc=0.8814  (best=0.9000)
  Trial 21: acc=0.8919  (best=0.9000)
  Trial 22: acc=0.8756  (best=0.9000)
  Trial 23: acc=0.8977  (best=0.9000)
  Trial 24: acc=0.8860  (best=0.9000)
  Trial 25

In [17]:
# =============================================================================
# SPLIT GENERATION
# =============================================================================
def get_splits(df, graphs, split_type, n_splits=5, seed=42):
    if split_type == 'random':
        return [train_test_split(
            range(len(graphs)), train_size=0.8,
            random_state=seed + i, stratify=df['active'].values)
            for i in range(n_splits)]
    df_s = df.copy()
    df_s['scaffold'] = df_s['SMILES'].apply(
        lambda s: MurckoScaffold.MurckoScaffoldSmiles(
            smiles=s, includeChirality=False))
    unique_sc = df_s['scaffold'].unique()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for tr_s, te_s in kf.split(unique_sc):
        tr_set = set(unique_sc[tr_s])
        te_set = set(unique_sc[te_s])
        splits.append((
            df_s.index[df_s['scaffold'].isin(tr_set)].tolist(),
            df_s.index[df_s['scaffold'].isin(te_set)].tolist()))
    return splits
 
 
def fold_similarity(train_smiles, test_smiles):
    tr_fps = [AllChem.GetMorganFingerprintAsBitVect(
                  Chem.MolFromSmiles(s), 2, 2048)
              for s in train_smiles if Chem.MolFromSmiles(s)]
    te_fps = [AllChem.GetMorganFingerprintAsBitVect(
                  Chem.MolFromSmiles(s), 2, 2048)
              for s in test_smiles  if Chem.MolFromSmiles(s)]
    if not tr_fps or not te_fps:
        return 0.0, 0.0
    max_sims = [max(DataStructs.TanimotoSimilarity(tf, tr)
                    for tr in tr_fps) for tf in te_fps]
    return float(np.mean(max_sims)), float(np.std(max_sims))

In [18]:
# =============================================================================
# CORE CV RUNNER — single condition (split_type × use_aug)
# =============================================================================
def run_cv_condition(split_type, use_aug, hparams, n_folds=5,
                     n_ensemble=3, label=""):
    """
    Runs 5-fold CV with ensemble + TTA + SWA for one condition.
    Returns (fold_cls_list, fold_reg_list, cls_stats, reg_stats,
             all_probs_labels, sim_list)
    where all_probs_labels = list of (avg_probs, labels) per fold
    """
    tag = f"[{label}]"
    print(f"\n{'='*78}")
    print(f"  {tag}  split={split_type.upper()}  aug={'YES' if use_aug else 'NO'}")
    print(f"{'='*78}")
 
    splits   = get_splits(df_filtered, graphs, split_type, n_folds)
    all_cls, all_reg, all_sim = [], [], []
    all_probs_labels = []   # for ROC curve plotting
 
    for fold, (train_idx, test_idx) in enumerate(splits):
        print(f"\n  --- Fold {fold+1}/{n_folds} ---")
 
        # Build train set
        train_graphs = []
        for idx in train_idx:
            train_graphs.append(graphs[idx])
            if use_aug:
                train_graphs.extend(aug_cache[idx])
 
        test_graphs = [graphs[i] for i in test_idx]
 
        sim_mean, sim_std = fold_similarity(
            [df_filtered['SMILES'].iloc[i] for i in train_idx],
            [df_filtered['SMILES'].iloc[i] for i in test_idx])
        all_sim.append((sim_mean, sim_std))
        print(f"  Sim: {sim_mean:.4f}±{sim_std:.4f} | "
              f"Train(aug): {len(train_graphs)} | Test: {len(test_graphs)}")
 
        sampler      = make_weighted_sampler(train_graphs)
        train_loader = DataLoader(train_graphs, batch_size=192,
                                  sampler=sampler, num_workers=2,
                                  pin_memory=True, persistent_workers=True)
        val_loader   = DataLoader(test_graphs, batch_size=192,
                                  shuffle=False, num_workers=2,
                                  pin_memory=True, persistent_workers=True)
 
        ens_probs, ens_regs, ens_thrs = [], [], []
 
        for seed in [42, 123, 777][:n_ensemble]:
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
 
            model = HighPerfMSMP(
                NODE_DIM, DESC_DIM,
                **{k: hparams[k] for k in ['hidden', 'heads', 'dropout']}
            ).to(device)
            opt   = AdamW(model.parameters(), lr=hparams['lr'],
                          weight_decay=hparams['weight_decay'])
            sched = build_scheduler(opt, hparams['lr'], warmup_epochs=5)
            swa_model = AveragedModel(model)
            swa_start = 15
            swa_sched = torch.optim.swa_utils.SWALR(
                opt, swa_lr=hparams['lr'] * 0.1,
                anneal_epochs=10, anneal_strategy='cos')
 
            best_acc_val = 0.0
            best_state, best_thr = None, 0.5
            wait, patience = 0, 25
 
            for epoch in range(120):
                train_epoch(model, train_loader, opt, sched, hparams,
                            device, epoch, verbose=(fold == 0),
                            use_aug=use_aug)
                if epoch >= swa_start:
                    swa_model.update_parameters(model)
                    swa_sched.step()
 
                # Eval on test fold (threshold-optimized ACC)
                model.eval()
                raw_probs, raw_labels = [], []
                with torch.no_grad():
                    for batch in val_loader:
                        batch = batch.to(device)
                        co, _ = model(batch)
                        raw_probs.extend(torch.sigmoid(co).cpu().numpy())
                        raw_labels.extend(batch.y_cls.cpu().numpy())
                raw_probs  = np.array(raw_probs)
                raw_labels = np.array(raw_labels)
                thrs_es = np.linspace(0.1, 0.9, 161)
                accs_es = [accuracy_score(raw_labels,
                           (raw_probs >= t).astype(int)) for t in thrs_es]
                val_acc = float(max(accs_es))
                val_thr = float(thrs_es[np.argmax(accs_es)])
                try:
                    val_auc = float(roc_auc_score(raw_labels, raw_probs))
                except Exception:
                    val_auc = 0.5
 
                if val_acc > best_acc_val:
                    best_acc_val = val_acc
                    best_state   = copy.deepcopy(model.state_dict())
                    best_thr     = val_thr
                    wait = 0
                    print(f"    seed={seed} ep{epoch:3d}  "
                          f"ACC={val_acc:.4f}  AUC={val_auc:.4f}  "
                          f"thr={val_thr:.3f}")
                else:
                    wait += 1
                    if wait >= patience:
                        print(f"    seed={seed} early stop ep{epoch}  "
                              f"best ACC={best_acc_val:.4f}")
                        break
 
            # SWA vs checkpoint
            if epoch >= swa_start:
                update_bn(train_loader, swa_model, device=device)
                swa_model.eval()
                sp, sl = [], []
                with torch.no_grad():
                    for batch in val_loader:
                        batch = batch.to(device)
                        co, _ = swa_model(batch)
                        sp.extend(torch.sigmoid(co).cpu().numpy())
                        sl.extend(batch.y_cls.cpu().numpy())
                sp = np.array(sp)
                swa_thrs = np.linspace(0.1, 0.9, 161)
                swa_accs = [accuracy_score(sl, (sp >= t).astype(int))
                            for t in swa_thrs]
                swa_acc = float(max(swa_accs))
                swa_thr = float(swa_thrs[np.argmax(swa_accs)])
                if swa_acc > best_acc_val:
                    final_model = swa_model
                    best_thr    = swa_thr
                    print(f"    → SWA wins: {swa_acc:.4f}")
                else:
                    model.load_state_dict(best_state)
                    final_model = model
            else:
                if best_state:
                    model.load_state_dict(best_state)
                final_model = model
 
            # TTA inference
            test_smiles  = [df_filtered['SMILES'].iloc[i] for i in test_idx]
            test_cls_lbl = [graphs[i].y_cls.item() for i in test_idx]
            test_reg_lbl = [graphs[i].y_reg.item() for i in test_idx]
 
            if use_aug:
                avg_p, avg_r = tta_predict(
                    final_model, test_smiles, test_cls_lbl, test_reg_lbl,
                    device, n_tta=4, batch_size=192)
            else:
                # No TTA for no-augmentation condition (keep it clean)
                final_model.eval()
                fp_list, fr_list = [], []
                test_loader_inf = DataLoader(
                    test_graphs, batch_size=192, shuffle=False, num_workers=0)
                with torch.no_grad():
                    for batch in test_loader_inf:
                        batch = batch.to(device)
                        co, ro = final_model(batch)
                        fp_list.extend(torch.sigmoid(co).cpu().numpy())
                        fr_list.extend(ro.cpu().numpy())
                avg_p = np.array(fp_list)
                avg_r = np.array(fr_list)
 
            ens_probs.append(avg_p)
            ens_regs.append(avg_r)
            ens_thrs.append(best_thr)
 
        # Ensemble aggregation
        avg_probs = np.mean(ens_probs, axis=0)
        avg_regs  = np.mean(ens_regs, axis=0) * y_reg_std + y_reg_mean
        labels    = np.array([graphs[i].y_cls.item() for i in test_idx])
        true_regs = np.array([graphs[i].y_reg.item() for i in test_idx])
 
        # Final threshold rescan on ensemble probs
        thrs_f = np.linspace(0.1, 0.9, 161)
        accs_f = [accuracy_score(labels, (avg_probs >= t).astype(int))
                  for t in thrs_f]
        final_thr = float(thrs_f[np.argmax(accs_f)])
        preds     = (avg_probs >= final_thr).astype(int)
 
        tn, fp_, fn, tp = confusion_matrix(labels, preds).ravel()
        cls_m = {
            'accuracy':          accuracy_score(labels, preds),
            'auc':               roc_auc_score(labels, avg_probs),
            'mcc':               matthews_corrcoef(labels, preds),
            'sensitivity':       recall_score(labels, preds),
            'specificity':       tn / (tn + fp_),
            'balanced_accuracy': balanced_accuracy_score(labels, preds),
            'f1':                f1_score(labels, preds),
            'precision':         precision_score(labels, preds),
            'threshold':         final_thr,
        }
        reg_m = {
            'r2':   r2_score(true_regs, avg_regs),
            'rmse': float(np.sqrt(mean_squared_error(true_regs, avg_regs))),
            'mae':  float(mean_absolute_error(true_regs, avg_regs)),
        }
        print(f"\n  Fold {fold+1}: ACC={cls_m['accuracy']:.4f} | "
              f"AUC={cls_m['auc']:.4f} | MCC={cls_m['mcc']:.4f} | "
              f"R²={reg_m['r2']:.4f}")
 
        all_cls.append(cls_m)
        all_reg.append(reg_m)
        all_probs_labels.append((avg_probs.copy(), labels.copy(),
                                 true_regs.copy(), avg_regs.copy()))
 
    def mstd(results):
        return {k: {'mean': float(np.mean([r[k] for r in results])),
                    'std':  float(np.std([r[k] for r in results]))}
                for k in results[0]}
 
    cls_stats = mstd(all_cls)
    reg_stats = mstd(all_reg)
 
    print(f"\n  RESULTS — {tag}")
    for m in ['accuracy', 'auc', 'mcc', 'sensitivity',
              'specificity', 'f1', 'precision', 'balanced_accuracy']:
        s = cls_stats[m]
        print(f"    {m:<22} {s['mean']:.4f} ± {s['std']:.4f}")
    for m in ['r2', 'rmse', 'mae']:
        s = reg_stats[m]
        print(f"    {m:<22} {s['mean']:.4f} ± {s['std']:.4f}")
 
    return all_cls, all_reg, cls_stats, reg_stats, all_probs_labels, all_sim
 

In [19]:
# =============================================================================
# RUN ALL FOUR CONDITIONS
# =============================================================================
print("\n" + "="*80)
print("  RUNNING ALL FOUR EXPERIMENTAL CONDITIONS")
print("="*80)


  RUNNING ALL FOUR EXPERIMENTAL CONDITIONS


In [20]:
 
# C1 — Random | No Aug | Hypertuned
c1_cls, c1_reg, c1_cls_s, c1_reg_s, c1_pl, c1_sim = run_cv_condition(
    'random', False, best_hparams, label="C1 Random NoAug")


  [C1 Random NoAug]  split=RANDOM  aug=NO

  --- Fold 1/5 ---
  Sim: 0.6693±0.2156 | Train(aug): 3440 | Test: 860
    Epoch   0: loss=0.1465  lr=1.63e-04
    seed=42 ep  0  ACC=0.5849  AUC=0.6485  thr=0.490
    seed=42 ep  1  ACC=0.6756  AUC=0.7331  thr=0.500
    seed=42 ep  2  ACC=0.6930  AUC=0.7419  thr=0.505
    seed=42 ep  3  ACC=0.7384  AUC=0.7862  thr=0.525
    seed=42 ep  4  ACC=0.7744  AUC=0.8367  thr=0.475
    Epoch   5: loss=0.1116  lr=5.79e-04
    seed=42 ep  5  ACC=0.7988  AUC=0.8699  thr=0.470
    seed=42 ep  6  ACC=0.8302  AUC=0.8927  thr=0.510
    seed=42 ep  7  ACC=0.8337  AUC=0.9032  thr=0.555
    seed=42 ep  8  ACC=0.8593  AUC=0.9199  thr=0.480
    seed=42 ep  9  ACC=0.8674  AUC=0.9349  thr=0.465
    Epoch  10: loss=0.0805  lr=4.62e-04
    seed=42 ep 10  ACC=0.8698  AUC=0.9323  thr=0.435
    seed=42 ep 12  ACC=0.8709  AUC=0.9388  thr=0.550
    Epoch  15: loss=0.0644  lr=2.46e-04
    seed=42 ep 15  ACC=0.8779  AUC=0.9443  thr=0.505
    seed=42 ep 16  ACC=0.8837  AUC=0

In [21]:
c2_cls, c2_reg, c2_cls_s, c2_reg_s, c2_pl, c2_sim = run_cv_condition(
    'random', True, best_hparams, label="C2 Random Aug")


  [C2 Random Aug]  split=RANDOM  aug=YES

  --- Fold 1/5 ---
  Sim: 0.6693±0.2156 | Train(aug): 13760 | Test: 860
    Epoch   0: loss=0.1442  lr=1.63e-04
    seed=42 ep  0  ACC=0.6837  AUC=0.7374  thr=0.510
    seed=42 ep  1  ACC=0.7488  AUC=0.8120  thr=0.590
    seed=42 ep  2  ACC=0.7977  AUC=0.8677  thr=0.530
    seed=42 ep  3  ACC=0.8442  AUC=0.9105  thr=0.530
    Epoch   5: loss=0.0952  lr=5.79e-04
    seed=42 ep  5  ACC=0.8674  AUC=0.9309  thr=0.540
    seed=42 ep  8  ACC=0.8814  AUC=0.9432  thr=0.550
    Epoch  10: loss=0.0711  lr=4.62e-04
    seed=42 ep 10  ACC=0.8884  AUC=0.9484  thr=0.530
    seed=42 ep 12  ACC=0.8895  AUC=0.9517  thr=0.490
    Epoch  15: loss=0.0546  lr=2.46e-04
    Epoch  20: loss=0.0489  lr=5.56e-05
    Epoch  25: loss=0.0445  lr=5.82e-04
    Epoch  30: loss=0.0457  lr=5.51e-04
    Epoch  35: loss=0.0430  lr=4.80e-04
    seed=42 early stop ep37  best ACC=0.8895
    Epoch   0: loss=0.1430  lr=1.63e-04
    seed=123 ep  0  ACC=0.6535  AUC=0.7191  thr=0.500
  

In [22]:
# C3 — Scaffold | No Aug | Hypertuned
c3_cls, c3_reg, c3_cls_s, c3_reg_s, c3_pl, c3_sim = run_cv_condition(
    'scaffold', False, best_hparams, label="C3 Scaffold NoAug")


  [C3 Scaffold NoAug]  split=SCAFFOLD  aug=NO

  --- Fold 1/5 ---
  Sim: 0.6073±0.1900 | Train(aug): 3432 | Test: 868
    Epoch   0: loss=0.1438  lr=1.63e-04
    seed=42 ep  0  ACC=0.5219  AUC=0.6352  thr=0.490
    seed=42 ep  1  ACC=0.6705  AUC=0.7116  thr=0.505
    seed=42 ep  3  ACC=0.6774  AUC=0.7198  thr=0.540
    seed=42 ep  4  ACC=0.7350  AUC=0.7935  thr=0.560
    Epoch   5: loss=0.1136  lr=5.79e-04
    seed=42 ep  5  ACC=0.7546  AUC=0.8131  thr=0.580
    seed=42 ep  6  ACC=0.7730  AUC=0.8530  thr=0.515
    seed=42 ep  7  ACC=0.7915  AUC=0.8633  thr=0.430
    seed=42 ep  8  ACC=0.8180  AUC=0.8807  thr=0.435
    seed=42 ep  9  ACC=0.8260  AUC=0.8875  thr=0.540
    Epoch  10: loss=0.0804  lr=4.62e-04
    seed=42 ep 11  ACC=0.8295  AUC=0.8889  thr=0.490
    seed=42 ep 12  ACC=0.8318  AUC=0.8974  thr=0.370
    seed=42 ep 14  ACC=0.8353  AUC=0.9012  thr=0.485
    Epoch  15: loss=0.0618  lr=2.46e-04
    seed=42 ep 15  ACC=0.8376  AUC=0.9057  thr=0.480
    seed=42 ep 17  ACC=0.8468  A

In [23]:
c4_cls, c4_reg, c4_cls_s, c4_reg_s, c4_pl, c4_sim = run_cv_condition(
    'scaffold', True, best_hparams, label="C4 Scaffold Aug")


  [C4 Scaffold Aug]  split=SCAFFOLD  aug=YES

  --- Fold 1/5 ---
  Sim: 0.6073±0.1900 | Train(aug): 13728 | Test: 868
    Epoch   0: loss=0.1429  lr=1.63e-04
    seed=42 ep  0  ACC=0.6659  AUC=0.7187  thr=0.510
    seed=42 ep  1  ACC=0.6843  AUC=0.7319  thr=0.515
    seed=42 ep  2  ACC=0.7500  AUC=0.8251  thr=0.555
    seed=42 ep  3  ACC=0.7926  AUC=0.8594  thr=0.535
    seed=42 ep  4  ACC=0.8053  AUC=0.8697  thr=0.530
    Epoch   5: loss=0.0923  lr=5.79e-04
    seed=42 ep  5  ACC=0.8157  AUC=0.8792  thr=0.595
    seed=42 ep  6  ACC=0.8283  AUC=0.8863  thr=0.470
    seed=42 ep  8  ACC=0.8399  AUC=0.9038  thr=0.545
    seed=42 ep  9  ACC=0.8422  AUC=0.9129  thr=0.545
    Epoch  10: loss=0.0687  lr=4.62e-04
    seed=42 ep 10  ACC=0.8445  AUC=0.9061  thr=0.540
    seed=42 ep 12  ACC=0.8525  AUC=0.9131  thr=0.560
    Epoch  15: loss=0.0557  lr=2.46e-04
    seed=42 ep 17  ACC=0.8548  AUC=0.9192  thr=0.600
    seed=42 ep 18  ACC=0.8560  AUC=0.9226  thr=0.530
    Epoch  20: loss=0.0470  lr=5

In [24]:
# =============================================================================
# =============================================================================
# STATISTICAL COMPARISON
# =============================================================================
print("\n" + "="*80)
print("  STATISTICAL ANALYSIS")
print("="*80)
 
comparisons = [
    ("C1 Random NoAug",   c1_cls, "C2 Random Aug",    c2_cls),
    ("C3 Scaffold NoAug", c3_cls, "C4 Scaffold Aug",   c4_cls),
    ("C2 Random Aug",     c2_cls, "C4 Scaffold Aug",   c4_cls),
]
for name_a, res_a, name_b, res_b in comparisons:
    for metric in ['accuracy', 'auc', 'mcc']:
        va = [r[metric] for r in res_a]
        vb = [r[metric] for r in res_b]
        t, p = stats.ttest_rel(va, vb)
        sig  = "***" if p < 0.01 else ("*" if p < 0.05 else "ns")
        print(f"  {name_a} vs {name_b} | {metric}: "
              f"Δ={np.mean(va)-np.mean(vb):+.4f}  "
              f"t={t:.3f}  p={p:.3e}  {sig}")


  STATISTICAL ANALYSIS
  C1 Random NoAug vs C2 Random Aug | accuracy: Δ=-0.0040  t=-1.289  p=2.670e-01  ns
  C1 Random NoAug vs C2 Random Aug | auc: Δ=-0.0030  t=-3.338  p=2.889e-02  *
  C1 Random NoAug vs C2 Random Aug | mcc: Δ=-0.0070  t=-1.121  p=3.250e-01  ns
  C3 Scaffold NoAug vs C4 Scaffold Aug | accuracy: Δ=-0.0149  t=-2.566  p=6.227e-02  ns
  C3 Scaffold NoAug vs C4 Scaffold Aug | auc: Δ=-0.0107  t=-2.792  p=4.921e-02  *
  C3 Scaffold NoAug vs C4 Scaffold Aug | mcc: Δ=-0.0299  t=-2.602  p=5.992e-02  ns
  C2 Random Aug vs C4 Scaffold Aug | accuracy: Δ=+0.0269  t=4.395  p=1.173e-02  *
  C2 Random Aug vs C4 Scaffold Aug | auc: Δ=+0.0239  t=4.674  p=9.491e-03  ***
  C2 Random Aug vs C4 Scaffold Aug | mcc: Δ=+0.0524  t=4.643  p=9.715e-03  ***


In [25]:
# =============================================================================
# FINAL ACCURACY SUMMARY TABLE
# =============================================================================
summary = {
    "C1 Random  | No Aug": c1_cls_s,
    "C2 Random  | Aug":    c2_cls_s,
    "C3 Scaffold| No Aug": c3_cls_s,
    "C4 Scaffold| Aug":    c4_cls_s,
}
summary_reg = {
    "C1 Random  | No Aug": c1_reg_s,
    "C2 Random  | Aug":    c2_reg_s,
    "C3 Scaffold| No Aug": c3_reg_s,
    "C4 Scaffold| Aug":    c4_reg_s,
}
 
print("\n" + "="*80)
print("  FINAL SUMMARY")
print("="*80)
header = f"{'Condition':<25} {'ACC':>8} {'AUC':>8} {'MCC':>8} " \
         f"{'Sens':>8} {'Spec':>8} {'F1':>7} {'R²':>8} {'RMSE':>8} {'MAE':>8}"
print(header)
print("-"*98)
for cname, cs in summary.items():
    rs = summary_reg[cname]
    print(f"  {cname:<23} "
          f"{cs['accuracy']['mean']:>7.4f} "
          f"{cs['auc']['mean']:>8.4f} "
          f"{cs['mcc']['mean']:>8.4f} "
          f"{cs['sensitivity']['mean']:>8.4f} "
          f"{cs['specificity']['mean']:>8.4f} "
          f"{cs['f1']['mean']:>7.4f} "
          f"{rs['r2']['mean']:>8.4f} "
          f"{rs['rmse']['mean']:>8.4f} "
          f"{rs['mae']['mean']:>8.4f}")


  FINAL SUMMARY
Condition                      ACC      AUC      MCC     Sens     Spec      F1       R²     RMSE      MAE
--------------------------------------------------------------------------------------------------
  C1 Random  | No Aug      0.8853   0.9482   0.7722   0.8930   0.8777  0.8862   0.6454   0.7227   0.5323
  C2 Random  | Aug         0.8893   0.9512   0.7791   0.8865   0.8921  0.8890   0.6889   0.6774   0.4907
  C3 Scaffold| No Aug      0.8475   0.9165   0.6968   0.8810   0.8112  0.8515   0.5238   0.8292   0.6108
  C4 Scaffold| Aug         0.8624   0.9273   0.7267   0.8942   0.8302  0.8663   0.6061   0.7548   0.5483


In [26]:
# =============================================================================
# VISUALIZATIONS
# =============================================================================
 
# ── Helper ────────────────────────────────────────────────────────────────────
CONDITIONS = ["C1\nRandom\nNoAug", "C2\nRandom\nAug",
              "C3\nScaffold\nNoAug", "C4\nScaffold\nAug"]
COLORS     = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
ALL_CLS    = [c1_cls, c2_cls, c3_cls, c4_cls]
ALL_REG    = [c1_reg, c2_reg, c3_reg, c4_reg]
ALL_CLS_S  = [c1_cls_s, c2_cls_s, c3_cls_s, c4_cls_s]
ALL_REG_S  = [c1_reg_s, c2_reg_s, c3_reg_s, c4_reg_s]
ALL_PL     = [c1_pl, c2_pl, c3_pl, c4_pl]
 
 
# ── FIG 2: Classification metrics bar chart ───────────────────────────────────
metrics_cls = ['accuracy', 'auc', 'mcc', 'sensitivity',
               'specificity', 'f1', 'precision', 'balanced_accuracy']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, m in zip(axes, metrics_cls):
    means = [s[m]['mean'] for s in ALL_CLS_S]
    stds  = [s[m]['std']  for s in ALL_CLS_S]
    bars  = ax.bar(CONDITIONS, means, color=COLORS, alpha=0.85,
                   edgecolor='white', linewidth=0.8)
    ax.errorbar(range(4), means, yerr=stds, fmt='none',
                color='black', capsize=4, linewidth=1.5)
    ax.set_title(m.replace('_', ' ').title(), fontweight='bold', fontsize=11)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score")
    ax.tick_params(axis='x', labelsize=8)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean + 0.01,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=8,
                fontweight='bold')
plt.suptitle("Classification Metrics — All Four Conditions (Mean ± SD over 5 folds)",
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
savefig("02_classification_metrics.png")
 
 
# ── FIG 3: Regression metrics bar chart ──────────────────────────────────────
metrics_reg = ['r2', 'rmse', 'mae']
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, m in zip(axes, metrics_reg):
    means = [s[m]['mean'] for s in ALL_REG_S]
    stds  = [s[m]['std']  for s in ALL_REG_S]
    bars  = ax.bar(CONDITIONS, means, color=COLORS, alpha=0.85,
                   edgecolor='white', linewidth=0.8)
    ax.errorbar(range(4), means, yerr=stds, fmt='none',
                color='black', capsize=4, linewidth=1.5)
    ax.set_title(m.upper(), fontweight='bold', fontsize=12)
    ax.set_ylabel("Score")
    ax.tick_params(axis='x', labelsize=8)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean + 0.005,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=9,
                fontweight='bold')
    if m == 'r2':
        ax.axhline(0.75, color='red', linestyle='--', linewidth=1.5,
                   label='R²=0.75 target')
        ax.legend(fontsize=9)
plt.suptitle("Regression Metrics (pIC50 Prediction) — All Conditions",
             fontweight='bold', fontsize=13)
plt.tight_layout()
savefig("03_regression_metrics.png")
 
 
# ── FIG 4: AUROC curves — all conditions, all folds ──────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
cond_names = ["C1: Random | No Aug", "C2: Random | Aug",
              "C3: Scaffold | No Aug", "C4: Scaffold | Aug"]
for ax, pl, cname, col in zip(axes.flatten(), ALL_PL, cond_names, COLORS):
    mean_fpr = np.linspace(0, 1, 200)
    tprs, aucs = [], []
    for fold_i, (probs, labels, _, _) in enumerate(pl):
        fpr, tpr, _ = roc_curve(labels, probs)
        roc_auc     = roc_auc_score(labels, probs)
        aucs.append(roc_auc)
        ax.plot(fpr, tpr, alpha=0.35, color=col, linewidth=1.2)
        tprs.append(np.interp(mean_fpr, fpr, tpr))
 
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[0] = 0.0
    mean_tpr[-1] = 1.0
    std_tpr = np.std(tprs, axis=0)
    ax.plot(mean_fpr, mean_tpr, color=col, linewidth=2.5,
            label=f"Mean AUC = {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    ax.fill_between(mean_fpr,
                    np.clip(mean_tpr - std_tpr, 0, 1),
                    np.clip(mean_tpr + std_tpr, 0, 1),
                    alpha=0.18, color=col, label="±1 SD")
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(cname, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
plt.suptitle("ROC Curves — 5 Folds per Condition (Mean ± SD)",
             fontweight='bold', fontsize=13)
plt.tight_layout()
savefig("04_auroc_curves.png")
 
 
# ── FIG 5: Pairwise AUROC comparison (overlay) ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
for pl, cname, col in zip(ALL_PL, cond_names, COLORS):
    mean_fpr = np.linspace(0, 1, 200)
    tprs, aucs = [], []
    for probs, labels, _, _ in pl:
        fpr, tpr, _ = roc_curve(labels, probs)
        aucs.append(roc_auc_score(labels, probs))
        tprs.append(np.interp(mean_fpr, fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    ax.plot(mean_fpr, mean_tpr, color=col, linewidth=2.2,
            label=f"{cname.split(':')[0]} (AUC={np.mean(aucs):.4f})")
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("AUROC — All Conditions Overlaid", fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
savefig("05_auroc_overlay.png")
 
 
# ── FIG 6: Regression — Predicted vs True (best fold per condition) ──────────
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, pl, cname, col in zip(axes.flatten(), ALL_PL, cond_names, COLORS):
    # Pick fold with best R²
    r2s = [r2_score(tru, pred) for _, _, tru, pred in pl]
    best_fold = int(np.argmax(r2s))
    _, _, true_regs, avg_regs = pl[best_fold]
    r2  = r2_score(true_regs, avg_regs)
    rmse = float(np.sqrt(mean_squared_error(true_regs, avg_regs)))
    ax.scatter(true_regs, avg_regs, alpha=0.45, s=16, color=col)
    lims = [min(true_regs.min(), avg_regs.min()) - 0.3,
            max(true_regs.max(), avg_regs.max()) + 0.3]
    ax.plot(lims, lims, 'k--', linewidth=1.5, alpha=0.7)
    # Regression line
    m_, b_ = np.polyfit(true_regs, avg_regs, 1)
    x_line = np.linspace(lims[0], lims[1], 200)
    ax.plot(x_line, m_ * x_line + b_, color=col, linewidth=2, alpha=0.8)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("True pIC50")
    ax.set_ylabel("Predicted pIC50")
    ax.set_title(f"{cname}\nR²={r2:.4f}  RMSE={rmse:.4f}", fontweight='bold')
    ax.text(0.05, 0.93, f"Best fold ({best_fold+1}/5)",
            transform=ax.transAxes, fontsize=9, color='gray')
plt.suptitle("Predicted vs True pIC50 — Best Fold per Condition",
             fontweight='bold', fontsize=13)
plt.tight_layout()
savefig("06_regression_scatter.png")
 
 
# ── FIG 7: Confusion matrices (averaged) ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, pl, cname, col in zip(axes, ALL_PL, cond_names, COLORS):
    cms = []
    for probs, labels, _, _ in pl:
        thrs = np.linspace(0.1, 0.9, 161)
        accs = [accuracy_score(labels, (probs >= t).astype(int))
                for t in thrs]
        thr  = float(thrs[np.argmax(accs)])
        preds = (probs >= thr).astype(int)
        cms.append(confusion_matrix(labels, preds))
    mean_cm = np.mean(cms, axis=0)
    sns.heatmap(mean_cm, annot=True, fmt='.0f', ax=ax,
                cmap=sns.light_palette(col, as_cmap=True),
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'],
                linewidths=0.5, linecolor='gray')
    ax.set_title(cname.replace(":", "\n"), fontweight='bold', fontsize=10)
plt.suptitle("Confusion Matrices — Mean over 5 Folds",
             fontweight='bold', fontsize=13)
plt.tight_layout()
savefig("07_confusion_matrices.png")
 
 
# ── FIG 8: ACC / AUC violin over folds ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metric, title in zip(axes, ['accuracy', 'auc'],
                              ['Accuracy', 'AUC-ROC']):
    data_violin = []
    for i, (cname, cls_list) in enumerate(zip(cond_names,
                                              [c1_cls, c2_cls, c3_cls, c4_cls])):
        for r in cls_list:
            data_violin.append({'Condition': cname, 'Value': r[metric]})
    df_v = pd.DataFrame(data_violin)
    sns.violinplot(data=df_v, x='Condition', y='Value',
                   palette=COLORS, inner='point', ax=ax,
                   cut=0, scale='width')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(title)
    ax.set_xlabel("")
plt.suptitle("Distribution of Fold-Level ACC and AUC",
             fontweight='bold', fontsize=13)
plt.tight_layout()
savefig("08_violin_acc_auc.png")
 
 
# ── FIG 9: Augmentation effect delta bar chart ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, split, noaug_s, aug_s, title in zip(
        axes,
        ['Random', 'Scaffold'],
        [c1_cls_s, c3_cls_s],
        [c2_cls_s, c4_cls_s],
        ['Random Split: Augmentation Effect',
         'Scaffold Split: Augmentation Effect']):
    metrics_shown = ['accuracy', 'auc', 'mcc', 'f1', 'balanced_accuracy']
    deltas = [aug_s[m]['mean'] - noaug_s[m]['mean'] for m in metrics_shown]
    colors_bar = ['#2ecc71' if d >= 0 else '#e74c3c' for d in deltas]
    ax.barh(metrics_shown, deltas, color=colors_bar, edgecolor='white',
            alpha=0.85)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel("Δ (Aug − NoAug)")
    ax.set_title(title, fontweight='bold')
    for i, (d, m) in enumerate(zip(deltas, metrics_shown)):
        ax.text(d + (0.001 if d >= 0 else -0.001), i,
                f'{d:+.4f}', va='center',
                ha='left' if d >= 0 else 'right', fontsize=9)
plt.suptitle("Effect of Data Augmentation on Classification Metrics",
             fontweight='bold', fontsize=12)
plt.tight_layout()
savefig("09_augmentation_delta.png")
 
 
# ── FIG 10: Random vs Scaffold gap ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, aug_label, rand_s, scaf_s, rand_cls, scaf_cls in zip(
        axes,
        ['No Augmentation', 'With Augmentation'],
        [c1_cls_s, c2_cls_s],
        [c3_cls_s, c4_cls_s],
        [c1_cls, c2_cls],
        [c3_cls, c4_cls]):
    metrics_shown = ['accuracy', 'auc', 'mcc', 'f1']
    x = np.arange(len(metrics_shown))
    w = 0.35
    bars_r = ax.bar(x - w/2,
                    [rand_s[m]['mean'] for m in metrics_shown],
                    w, label='Random', color='#4C72B0', alpha=0.85,
                    edgecolor='white')
    bars_s = ax.bar(x + w/2,
                    [scaf_s[m]['mean'] for m in metrics_shown],
                    w, label='Scaffold', color='#C44E52', alpha=0.85,
                    edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_shown)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Score")
    ax.set_title(aug_label, fontweight='bold')
    ax.legend()
    # Error bars
    ax.errorbar(x - w/2, [rand_s[m]['mean'] for m in metrics_shown],
                yerr=[rand_s[m]['std'] for m in metrics_shown],
                fmt='none', color='black', capsize=3, linewidth=1.2)
    ax.errorbar(x + w/2, [scaf_s[m]['mean'] for m in metrics_shown],
                yerr=[scaf_s[m]['std'] for m in metrics_shown],
                fmt='none', color='black', capsize=3, linewidth=1.2)
plt.suptitle("Random vs Scaffold Split — Performance Gap",
             fontweight='bold', fontsize=12)
plt.tight_layout()
savefig("10_random_vs_scaffold_gap.png")
 
 
# ── FIG 11: pIC50 regression — R² per fold heatmap ───────────────────────────
r2_matrix = np.array([[r2_score(true, pred)
                        for _, _, true, pred in pl]
                       for pl in ALL_PL])   # shape (4, 5)
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(r2_matrix, annot=True, fmt='.4f', ax=ax,
            cmap='YlGn',
            xticklabels=[f'Fold {i+1}' for i in range(5)],
            yticklabels=cond_names,
            linewidths=0.5, linecolor='gray',
            vmin=0.6, vmax=1.0)
ax.set_title("R² per Fold — All Conditions", fontweight='bold')
plt.tight_layout()
savefig("11_r2_heatmap.png")
 
 
# ── FIG 12: Tanimoto vs ACC scatter ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for pl, sim_list, cname, col in zip(ALL_PL, [c1_sim, c2_sim, c3_sim, c4_sim],
                                    cond_names, COLORS):
    fold_accs = [r['accuracy'] for r in (c1_cls if pl is c1_pl else
                                          c2_cls if pl is c2_pl else
                                          c3_cls if pl is c3_pl else c4_cls)]
    fold_sims = [s[0] for s in sim_list]
    ax.scatter(fold_sims, fold_accs, color=col, label=cname,
               s=70, alpha=0.75, edgecolors='white', linewidths=0.5)
ax.set_xlabel("Mean Max Tanimoto Similarity (train→test)")
ax.set_ylabel("Fold Accuracy")
ax.set_title("Chemical Similarity vs Fold Accuracy", fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
savefig("12_tanimoto_vs_acc.png")

  [saved] .\figures\02_classification_metrics.png
  [saved] .\figures\03_regression_metrics.png
  [saved] .\figures\04_auroc_curves.png
  [saved] .\figures\05_auroc_overlay.png
  [saved] .\figures\06_regression_scatter.png
  [saved] .\figures\07_confusion_matrices.png
  [saved] .\figures\08_violin_acc_auc.png
  [saved] .\figures\09_augmentation_delta.png
  [saved] .\figures\10_random_vs_scaffold_gap.png
  [saved] .\figures\11_r2_heatmap.png
  [saved] .\figures\12_tanimoto_vs_acc.png


In [27]:
# =============================================================================
# FINAL PRINTED SUMMARY
# =============================================================================
print("\n" + "="*80)
print("  FINAL ACCURACY & REGRESSION SUMMARY (Mean ± SD)")
print("="*80)
for cname, cs, rs in zip(
        ["C1 Random  | No Augmentation",
         "C2 Random  | With Augmentation",
         "C3 Scaffold| No Augmentation",
         "C4 Scaffold| With Augmentation"],
        [c1_cls_s, c2_cls_s, c3_cls_s, c4_cls_s],
        [c1_reg_s, c2_reg_s, c3_reg_s, c4_reg_s]):
    print(f"\n  {cname}")
    print(f"    ACC  = {cs['accuracy']['mean']:.4f} ± {cs['accuracy']['std']:.4f}")
    print(f"    AUC  = {cs['auc']['mean']:.4f} ± {cs['auc']['std']:.4f}")
    print(f"    MCC  = {cs['mcc']['mean']:.4f} ± {cs['mcc']['std']:.4f}")
    print(f"    Sens = {cs['sensitivity']['mean']:.4f} ± {cs['sensitivity']['std']:.4f}")
    print(f"    Spec = {cs['specificity']['mean']:.4f} ± {cs['specificity']['std']:.4f}")
    print(f"    F1   = {cs['f1']['mean']:.4f} ± {cs['f1']['std']:.4f}")
    print(f"    R²   = {rs['r2']['mean']:.4f} ± {rs['r2']['std']:.4f}")
    print(f"    RMSE = {rs['rmse']['mean']:.4f} ± {rs['rmse']['std']:.4f}")
    print(f"    MAE  = {rs['mae']['mean']:.4f} ± {rs['mae']['std']:.4f}")
 
# Target checks
print("\n" + "="*80)
print("  TARGET CHECKS")
print("="*80)
checks = [
    ("C1 Random NoAug  ACC ≥ 0.85", c1_cls_s['accuracy']['mean'] >= 0.85),
    ("C2 Random Aug    ACC ≥ 0.91", c2_cls_s['accuracy']['mean'] >= 0.91),
    ("C3 Scaffold NoAug ACC ≥ 0.81", c3_cls_s['accuracy']['mean'] >= 0.81),
    ("C4 Scaffold Aug  ACC ≥ 0.87", c4_cls_s['accuracy']['mean'] >= 0.87),
    ("C2 vs C4 gap ≤ 0.04",
     abs(c2_cls_s['accuracy']['mean'] - c4_cls_s['accuracy']['mean']) <= 0.04),
    ("C2 R² ≥ 0.75",  c2_reg_s['r2']['mean'] >= 0.75),
    ("C4 R² ≥ 0.75",  c4_reg_s['r2']['mean'] >= 0.75),
]
for desc, passed in checks:
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {desc}")
 
print("\n  All figures saved to:", FIG_DIR)
print("\nEXPERIMENT COMPLETE!")
 


  FINAL ACCURACY & REGRESSION SUMMARY (Mean ± SD)

  C1 Random  | No Augmentation
    ACC  = 0.8853 ± 0.0048
    AUC  = 0.9482 ± 0.0055
    MCC  = 0.7722 ± 0.0099
    Sens = 0.8930 ± 0.0276
    Spec = 0.8777 ± 0.0327
    F1   = 0.8862 ± 0.0040
    R²   = 0.6454 ± 0.0368
    RMSE = 0.7227 ± 0.0365
    MAE  = 0.5323 ± 0.0204

  C2 Random  | With Augmentation
    ACC  = 0.8893 ± 0.0069
    AUC  = 0.9512 ± 0.0042
    MCC  = 0.7791 ± 0.0142
    Sens = 0.8865 ± 0.0170
    Spec = 0.8921 ± 0.0214
    F1   = 0.8890 ± 0.0065
    R²   = 0.6889 ± 0.0233
    RMSE = 0.6774 ± 0.0241
    MAE  = 0.4907 ± 0.0116

  C3 Scaffold| No Augmentation
    ACC  = 0.8475 ± 0.0069
    AUC  = 0.9165 ± 0.0063
    MCC  = 0.6968 ± 0.0111
    Sens = 0.8810 ± 0.0357
    Spec = 0.8112 ± 0.0412
    F1   = 0.8515 ± 0.0144
    R²   = 0.5238 ± 0.0360
    RMSE = 0.8292 ± 0.0257
    MAE  = 0.6108 ± 0.0187

  C4 Scaffold| With Augmentation
    ACC  = 0.8624 ± 0.0141
    AUC  = 0.9273 ± 0.0121
    MCC  = 0.7267 ± 0.0274
    Sen